# Comprehensive Transformer Architecture Implementation - Interview Practice

## Overview
This notebook covers the complete implementation of transformer architectures from scratch, including:
- Core attention mechanisms (single-head, multi-head, MQA, GQA)
- Positional encodings (sinusoidal, RoPE)
- Normalization techniques (LayerNorm, RMSNorm)
- Advanced features (KV Cache, MoE, Quantization)
- Encoder, Decoder, and Encoder-Decoder architectures

**Dataset**: We'll build a character-level language model on a small text corpus.

**Goal**: Master every component of modern transformer architectures for technical interviews.

## Part 1: Setup and Data Preparation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from typing import Optional, Tuple
from torch.utils.data import Dataset, DataLoader

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# TODO: Load a simple text dataset (e.g., tiny shakespeare or create synthetic data)
# TODO: Create character-level vocabulary
# TODO: Implement data preprocessing and create train/val splits
# TODO: Create DataLoader

## Part 2: Single Head Attention

Implement the basic scaled dot-product attention mechanism:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Initialize Q, K, V projection layers
        # TODO: Initialize output projection
        # TODO: Store d_model and create dropout layer
        pass
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Project to Q, K, V
        # TODO: Compute attention scores (Q @ K.T / sqrt(d_k))
        # TODO: Apply mask if provided (set masked positions to -inf before softmax)
        # TODO: Apply softmax and dropout
        # TODO: Multiply by V and project output
        pass

## Part 3: Multi-Head Attention

Extend to multiple attention heads operating in parallel.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Assert d_model is divisible by num_heads
        # TODO: Store d_model, num_heads, d_k (d_model // num_heads)
        # TODO: Initialize Q, K, V projections
        # TODO: Initialize output projection and dropout
        pass
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Get batch_size and seq_len
        # TODO: Project to Q, K, V and reshape to (batch, num_heads, seq_len, d_k)
        # TODO: Compute scaled dot-product attention
        # TODO: Apply mask if provided
        # TODO: Apply softmax and dropout
        # TODO: Multiply by V
        # TODO: Concatenate heads and project output
        pass

## Part 4: Attention Masks

Implement different types of masks for attention.

In [ ]:
def create_padding_mask(seq: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    """Create mask for padding tokens (1 for valid, 0 for padding)"""
    # TODO: Return mask of shape (batch_size, 1, 1, seq_len)
    pass

def create_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """Create causal mask for autoregressive generation (lower triangular)"""
    # TODO: Return lower triangular mask of shape (1, 1, seq_len, seq_len)
    pass

def combine_masks(padding_mask: Optional[torch.Tensor], 
                  causal_mask: Optional[torch.Tensor]) -> Optional[torch.Tensor]:
    """Combine padding and causal masks"""
    # TODO: Combine masks using logical AND
    pass

## Part 5: Feed-Forward Network

Position-wise fully connected feed-forward network.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Initialize two linear layers (d_model -> d_ff -> d_model)
        # TODO: Initialize dropout
        pass
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Linear -> ReLU/GELU -> Dropout -> Linear
        pass

## Part 6: Normalization Layers

### 6.1 Layer Normalization

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        # TODO: Initialize learnable parameters (gamma, beta)
        # TODO: Store eps
        pass
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Compute mean and std along last dimension
        # TODO: Normalize: (x - mean) / (std + eps)
        # TODO: Apply scale (gamma) and shift (beta)
        pass

### 6.2 RMS Normalization

Used in LLaMA and other modern models.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        # TODO: Initialize weight parameter
        # TODO: Store eps
        pass
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Compute RMS: sqrt(mean(x^2) + eps)
        # TODO: Normalize and scale
        pass

## Part 7: Positional Encodings

### 7.1 Sinusoidal Positional Encoding

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        # TODO: Create PE matrix using sin/cos
        # TODO: PE[pos, 2i] = sin(pos / 10000^(2i/d_model))
        # TODO: PE[pos, 2i+1] = cos(pos / 10000^(2i/d_model))
        # TODO: Register as buffer
        pass
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Add positional encoding to input
        pass

### 7.2 Rotary Positional Embeddings (RoPE)

Used in models like LLaMA, GPT-NeoX.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        # TODO: Compute inverse frequencies
        # TODO: Create rotation matrices
        pass
    
    def forward(self, q: torch.Tensor, k: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # TODO: Apply rotation to q and k
        # TODO: Rotate pairs of dimensions
        pass

## Part 8: Transformer Blocks

### 8.1 Encoder Block

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, 
                 dropout: float = 0.1, use_rms_norm: bool = False):
        super().__init__()
        # TODO: Initialize multi-head attention
        # TODO: Initialize feed-forward network
        # TODO: Initialize normalization layers (LayerNorm or RMSNorm)
        # TODO: Initialize dropout layers
        pass
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Self-attention with pre-norm and residual
        # TODO: Feed-forward with pre-norm and residual
        pass

### 8.2 Decoder Block with Cross-Attention

In [ ]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Initialize masked self-attention
        # TODO: Initialize cross-attention
        # TODO: Initialize feed-forward network
        # TODO: Initialize 3 normalization layers
        # TODO: Initialize dropout layers
        pass
    
    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Masked self-attention
        # TODO: Cross-attention (Q from decoder, K,V from encoder)
        # TODO: Feed-forward
        pass

## Part 9: Efficient Attention Variants

### 9.1 Multi-Query Attention (MQA)

Single K,V head shared across all Q heads.

In [ ]:
class MultiQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Multiple Q heads, single K and V
        # TODO: Initialize projections
        pass
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Project Q with multiple heads
        # TODO: Project K, V with single head, broadcast to all Q heads
        # TODO: Compute attention
        pass

### 9.2 Grouped-Query Attention (GQA)

Groups of Q heads share K,V heads (middle ground between MHA and MQA).

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Assert num_heads is divisible by num_kv_heads
        # TODO: Initialize Q projection (num_heads)
        # TODO: Initialize K, V projections (num_kv_heads)
        pass
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Project Q, K, V
        # TODO: Repeat K, V heads to match Q heads
        # TODO: Compute attention
        pass

## Part 10: KV Cache

Cache key-value pairs for efficient autoregressive generation.

In [ ]:
class MultiHeadAttentionWithCache(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Initialize like regular MHA
        pass
    
    def forward(self, x: torch.Tensor, 
                cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
                mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        # TODO: Project Q, K, V
        # TODO: If cache exists, concatenate with cached K, V
        # TODO: Compute attention
        # TODO: Return output and updated cache (K, V)
        pass

## Part 11: Latent Attention

Compress K, V to lower-dimensional latent space.

In [ ]:
class LatentAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_latents: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Initialize latent queries
        # TODO: Initialize cross-attention to compress input
        # TODO: Initialize regular attention in latent space
        # TODO: Initialize projection back to original space
        pass
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Compress input to latent space
        # TODO: Apply attention in latent space
        # TODO: Project back to original dimension
        pass

## Part 12: Quantization

### 12.1 Symmetric Quantization

In [ ]:
def quantize_tensor(tensor: torch.Tensor, num_bits: int = 8) -> Tuple[torch.Tensor, float]:
    """Quantize tensor to int representation"""
    # TODO: Compute scale factor
    # TODO: Quantize to integers
    # TODO: Return quantized tensor and scale
    pass

def dequantize_tensor(quantized: torch.Tensor, scale: float) -> torch.Tensor:
    """Dequantize back to float"""
    # TODO: Multiply by scale
    pass

class QuantizedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, num_bits: int = 8):
        super().__init__()
        # TODO: Initialize weight and bias
        # TODO: Quantize weight during initialization
        pass
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Dequantize weight
        # TODO: Apply linear transformation
        pass

## Part 13: Mixture of Experts (MoE)

### 13.1 Single-Gate MoE

In [ ]:
class Expert(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Create a simple FFN expert
        pass
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Apply FFN
        pass

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model: int, d_ff: int, num_experts: int, 
                 top_k: int = 2, dropout: float = 0.1):
        super().__init__()
        # TODO: Create multiple expert networks
        # TODO: Create gating network
        # TODO: Store top_k
        pass
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Compute gating scores
        # TODO: Select top-k experts per token
        # TODO: Route tokens to selected experts
        # TODO: Combine expert outputs with gating weights
        pass

### 13.2 Multi-Gate MoE

Different gates for different tasks/modalities.

In [ ]:
class MultiGateMixtureOfExperts(nn.Module):
    def __init__(self, d_model: int, d_ff: int, num_experts: int, 
                 num_gates: int = 2, top_k: int = 2, dropout: float = 0.1):
        super().__init__()
        # TODO: Create shared expert networks
        # TODO: Create multiple gating networks
        # TODO: Create gate selection mechanism
        pass
    
    def forward(self, x: torch.Tensor, gate_idx: int = 0) -> torch.Tensor:
        # TODO: Select appropriate gate
        # TODO: Route through experts
        pass

## Part 14: Learning Rate Schedulers

In [ ]:
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps: int, total_steps: int, 
                 min_lr: float = 1e-6, max_lr: float = 1e-3):
        # TODO: Store parameters
        pass
    
    def step(self, current_step: int):
        # TODO: Implement warmup phase (linear increase)
        # TODO: Implement cosine decay phase
        pass

class InverseSqrtScheduler:
    def __init__(self, optimizer, warmup_steps: int, d_model: int):
        # TODO: Implement scheduler from "Attention is All You Need"
        # lr = d_model^(-0.5) * min(step^(-0.5), step * warmup_steps^(-1.5))
        pass
    
    def step(self, current_step: int):
        # TODO: Update learning rate
        pass

## Part 15: Complete Model Assembly

### 15.1 Decoder-Only Language Model (GPT-style)

In [ ]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, 
                 num_layers: int, d_ff: int, max_len: int = 512, 
                 dropout: float = 0.1, use_rope: bool = False,
                 use_moe: bool = False, use_gqa: bool = False):
        super().__init__()
        # TODO: Token embedding
        # TODO: Positional encoding (sinusoidal or RoPE)
        # TODO: Stack of transformer decoder blocks
        # TODO: Output projection to vocabulary
        pass
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Embed tokens
        # TODO: Add positional encoding
        # TODO: Apply transformer blocks
        # TODO: Project to vocabulary
        pass
    
    def generate(self, prompt: torch.Tensor, max_new_tokens: int, 
                 temperature: float = 1.0, top_k: Optional[int] = None) -> torch.Tensor:
        # TODO: Implement autoregressive generation
        # TODO: Use KV cache if available
        # TODO: Apply temperature and top-k sampling
        pass

### 15.2 Encoder-Decoder Model (T5-style)

In [ ]:
class TransformerEncoderDecoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_encoder_layers: int, num_decoder_layers: int,
                 d_ff: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        # TODO: Encoder and decoder embeddings
        # TODO: Positional encodings
        # TODO: Encoder stack
        # TODO: Decoder stack with cross-attention
        # TODO: Output projection
        pass
    
    def encode(self, src: torch.Tensor, src_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Encode source sequence
        pass
    
    def decode(self, tgt: torch.Tensor, encoder_output: torch.Tensor,
               src_mask: Optional[torch.Tensor] = None,
               tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Decode target sequence with cross-attention
        pass
    
    def forward(self, src: torch.Tensor, tgt: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # TODO: Full forward pass
        pass

## Part 16: Training Loop

In [ ]:
# TODO: Initialize model with chosen architecture
# TODO: Define loss function (CrossEntropyLoss)
# TODO: Define optimizer (AdamW)
# TODO: Define scheduler
# TODO: Implement training loop with:
#   - Forward pass
#   - Loss computation
#   - Backward pass
#   - Gradient clipping
#   - Optimizer step
#   - Scheduler step
#   - Logging
# TODO: Track perplexity

## Part 17: Evaluation and Generation

In [ ]:
# TODO: Implement evaluation function
# TODO: Compute validation loss and perplexity
# TODO: Implement text generation function
# TODO: Test different sampling strategies (greedy, top-k, nucleus)

## Part 18: Experiments and Comparisons

### Experiment 1: Compare Attention Mechanisms

In [ ]:
# TODO: Train models with MHA, MQA, and GQA
# TODO: Compare training speed and memory usage
# TODO: Compare final performance

### Experiment 2: Compare Positional Encodings

In [ ]:
# TODO: Train with sinusoidal vs RoPE
# TODO: Test on sequences longer than training length
# TODO: Compare generalization

### Experiment 3: Evaluate MoE

In [ ]:
# TODO: Compare dense model vs MoE with same total parameters
# TODO: Analyze expert specialization
# TODO: Measure inference speed

### Experiment 4: Quantization Impact

In [ ]:
# TODO: Quantize trained model to 8-bit and 4-bit
# TODO: Measure performance degradation
# TODO: Measure memory savings

## Part 19: Interview Questions to Prepare

After implementing the above, be ready to answer:

1. **Architecture Questions**:
   - Why do we scale attention scores by sqrt(d_k)?
   - What's the difference between encoder and decoder self-attention?
   - How does cross-attention work in encoder-decoder models?
   - Why use pre-norm vs post-norm in transformer blocks?

2. **Efficiency Questions**:
   - How does MQA reduce memory compared to MHA?
   - Explain KV cache and its benefits
   - What are the tradeoffs of GQA vs MQA vs MHA?
   - How does quantization work and what's the quality-size tradeoff?

3. **Training Questions**:
   - Why use warmup in learning rate scheduling?
   - Explain gradient clipping and why it's important
   - What causes instability in training large transformers?
   - How does dropout help in transformers?

4. **Advanced Topics**:
   - Compare sinusoidal PE vs RoPE - which is better for long sequences?
   - How does MoE achieve better parameter efficiency?
   - What is the load balancing problem in MoE?
   - Explain latent attention and its computational benefits